<a href="https://colab.research.google.com/github/sarabdar/pytorch/blob/main/Logistic_Regression_Cancer_using_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install kaggle

In [37]:
%env KAGGLE_USERNAME="YOUR NAME"
%env KAGGLE_KEY="YOUR KEY"

env: KAGGLE_USERNAME="YOUR NAME"
env: KAGGLE_KEY="YOUR KEY"


In [7]:
import kaggle


dataset = "marshuu/breast-cancer"

kaggle.api.dataset_download_files(dataset,  path= "/content")

print("Download complete.")

Dataset URL: https://www.kaggle.com/datasets/marshuu/breast-cancer
Download complete.


In [8]:
!unzip /content/breast-cancer.zip -d /content/breast_cancer_data

Archive:  /content/breast-cancer.zip
  inflating: /content/breast_cancer_data/breast_cancer.csv  


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [9]:
df = pd.read_csv('/content/breast_cancer_data/breast_cancer.csv')
df.head()

,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,5,1,1,1,2,1,3,1,1,2
1,5,4,4,5,7,10,3,2,1,2
2,3,1,1,1,2,2,3,1,1,2
3,6,8,8,1,3,4,3,7,1,2
4,4,1,1,3,2,1,3,1,1,2


In [10]:
df["Class"].value_counts().to_frame()

,count
Class,
2,444
4,239


### Here:

2 is for Benign

4 is for Malignant

### Class 2: Benign
When a sample is labeled as 2, it means the cell characteristics observed (like clump thickness and uniformity) point toward a non-cancerous growth.


### Class 4: Malignant
When a sample is labeled as 4, it indicates cancerous cells. These samples typically have much higher values in the independent variables, such as high "Clump Thickness" or frequent "Mitoses."

In [6]:
# Create the data class adapted for breast cancer dataset
class BreastCancerData(Dataset):

    # Constructor
    def __init__(self, filepath='/content/breast_cancer_data/breast_cancer.csv'):
        # Load the CSV file
        df = pd.read_csv(filepath)

        # Separate features (X) and target (y)
        # All columns except 'Class' are features
        X = df.drop('Class', axis=1).values
        # Map 'Class' values: 2 (Benign) to 0, 4 (Malignant) to 1
        y = (df['Class'] == 4).astype(int).values

        # Convert numpy arrays to PyTorch tensors
        self.x = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1) # Reshape y to (num_samples, 1)

        # Store the length of the dataset
        self.len = self.x.shape[0]

    # Getter
    def __getitem__(self, index):
        return self.x[index], self.y[index]

    # Get Length
    def __len__(self):
        return self.len

Now that we have our `BreastCancerData` class, we can create an instance of it and then use `DataLoader` to iterate over our data in batches.

In [28]:
# Create an instance of our custom dataset
breast_cancer_dataset = BreastCancerData()

# Create a DataLoader
# batch_size: how many samples per batch to load
# shuffle: set to True to have the data reshuffled at every epoch
# num_workers: how many subprocesses to use for data loading. 0 means that the data will be loaded in the main process.


criterion_rms = nn.MSELoss()
learning_rate = 2
batch_size = 3
breast_cancer_dataloader = DataLoader(dataset=breast_cancer_dataset, batch_size=batch_size, shuffle=True)

# Let's inspect one batch of data
for i, (features, labels) in enumerate(breast_cancer_dataloader):
    print(f"Batch {i+1}:")
    print(f"Features shape: {features.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"First 5 features in batch: {features[:5]}")
    print(f"First 5 labels in batch: {labels[:5]}")
    break # Just show the first batch

Batch 1:
Features shape: torch.Size([3, 9])
Labels shape: torch.Size([3, 1])
First 5 features in batch: tensor([[10.,  5.,  7.,  3.,  3.,  7.,  3.,  3.,  8.],
        [ 4.,  2.,  1.,  1.,  2.,  2.,  3.,  1.,  1.],
        [ 4.,  1.,  4.,  1.,  2.,  1.,  1.,  1.,  1.]])
First 5 labels in batch: tensor([[1.],
        [0.],
        [0.]])


### Now lets create model class

In [29]:
class logistic_regression(nn.Module):
  def __init__(self, n_inputs):
    super(logistic_regression, self).__init__()
    self.linear = nn.Linear(n_inputs, 1)

  def forward(self, x):
    y_pred = torch.sigmoid(self.linear(x))
    return y_pred

In [30]:
model = logistic_regression(breast_cancer_dataset.x.shape[1])

In [31]:
model.state_dict()

OrderedDict([('linear.weight',
              tensor([[ 0.1907, -0.1240,  0.0224, -0.1012, -0.2495, -0.0762, -0.1964, -0.2239,
                        0.2734]])),
             ('linear.bias', tensor([-0.0569]))])

In [32]:
model.state_dict()['linear.weight'].data[0]

tensor([ 0.1907, -0.1240,  0.0224, -0.1012, -0.2495, -0.0762, -0.1964, -0.2239,
         0.2734])

In [33]:
# Let set the weight and bias instead of Random

model.state_dict() ['linear.weight'].data[0] = torch.tensor([[0.12, 0.17, 0.12, 0.16, 0.013, 0.065, 0.087, 0.15, 0.19]])
model.state_dict() ['linear.bias'].data[0] = torch.tensor([[-0.12]])
print("The parameters: ", model.state_dict())

The parameters:  OrderedDict({'linear.weight': tensor([[0.1200, 0.1700, 0.1200, 0.1600, 0.0130, 0.0650, 0.0870, 0.1500, 0.1900]]), 'linear.bias': tensor([-0.1200])})


In [34]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [35]:
# Train the model

def train_model(epochs):
    for epoch in range(epochs):
        for x, y in breast_cancer_dataloader:
            yhat = model(x)
            loss = criterion_rms(yhat, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

train_model(100)

In [36]:
# Make the Prediction

yhat = model(breast_cancer_dataset.x)
label = yhat > 0.5
print("The accuracy: ", torch.mean((label == breast_cancer_dataset.y.type(torch.ByteTensor)).type(torch.float)))

The accuracy:  tensor(0.6501)
